# AwaDB
>[AwaDB](https://github.com/awa-ai/awadb) 是一个原生AI数据库，用于存储和搜索由LLM应用程序使用的嵌入向量。

您需要安装 `langchain-community` 才能使用此集成，命令为 `pip install -qU langchain-community`

本笔记本展示了如何使用与 `AwaDB` 相关的各项功能。

In [ ]:
%pip install --upgrade --quiet  awadb

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import AwaDB
from langchain_text_splitters import CharacterTextSplitter

In [ ]:
loader = TextLoader("../../how_to/state_of_the_union.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=0)
docs = text_splitter.split_documents(documents)

In [ ]:
db = AwaDB.from_documents(docs)
query = "What did the president say about Ketanji Brown Jackson"
docs = db.similarity_search(query)

In [4]:
print(docs[0].page_content)

And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.


## 相似度搜索与评分

This guide explains how to perform similarity searches and retrieve scores for the results.

本指南将介绍如何执行相似度搜索并获取结果的评分。

**Prerequisites**

*   A Milvus instance running.
*   A collection with data loaded.
*   Familiarity with Milvus concepts.

**前提条件**

*   正在运行的 Milvus 实例。
*   已加载数据的集合。
*   熟悉 Milvus 的概念。

**Steps**

1.  **Import necessary libraries.**

    ```python
    from pymilvus import connections, Collection
    ```

    **步骤**

    1.  **导入必要的库。**

        ```python
        from pymilvus import connections, Collection
        ```

2.  **Connect to Milvus.**

    ```python
    connections.connect("default", host="localhost", port="19530")
    ```

    **2. 连接到 Milvus。**

    ```python
    connections.connect("default", host="localhost", port="19530")
    ```

3.  **Get the collection.**

    ```python
    collection = Collection("your_collection_name")
    ```

    **3. 获取集合。**

    ```python
    collection = Collection("your_collection_name")
    ```

4.  **Load the collection into memory.**

    ```python
    collection.load()
    ```

    **4. 将集合加载到内存。**

    ```python
    collection.load()
    ```

5.  **Perform a similarity search.**

    The `search` method allows you to perform similarity searches. You need to provide the search parameters, including the search vector, the search field, the metric type, and the number of results to return (`limit`).

    To get the similarity scores, you need to set `output_fields=["*"]` or specify the fields you want to retrieve. The scores are returned in the `scores` field of the search results.

    ```python
    search_params = {
        "metric_type": "L2",
        "params": {"nprobe": 10},
    }

    results = collection.search(
        data=[[1.0, 2.0, 3.0]],  # Replace with your actual search vector
        anns_field="embedding",  # Replace with your embedding field name
        param=search_params,
        limit=3,
        expr=None,
        output_fields=["*"]  # Retrieve all fields, including scores
    )
    ```

    **5. 执行相似度搜索。**

    `search` 方法允许您执行相似度搜索。您需要提供搜索参数，包括搜索向量、搜索字段、度量类型以及要返回的结果数量（`limit`）。

    要获取相似度评分，您需要设置 `output_fields=["*"]` 或指定要检索的字段。评分将作为 `scores` 字段返回在搜索结果中。

    ```python
    search_params = {
        "metric_type": "L2",
        "params": {"nprobe": 10},
    }

    results = collection.search(
        data=[[1.0, 2.0, 3.0]],  # 请替换为您实际的搜索向量
        anns_field="embedding",  # 请替换为您嵌入字段的名称
        param=search_params,
        limit=3,
        expr=None,
        output_fields=["*"]  # 检索所有字段，包括评分
    )
    ```

6.  **Process the search results.**

    The `results` object contains the search hits. You can iterate through the hits and access the entity data and scores.

    ```python
    for hit in results[0]:
        print(f"Hit ID: {hit.id}, Score: {hit.score}, Entity: {hit.entity}")
    ```

    **6. 处理搜索结果。**

    `results` 对象包含搜索命中。您可以遍历命中并访问实体数据和评分。

    ```python
    for hit in results[0]:
        print(f"命中 ID: {hit.id}, 评分: {hit.score}, 实体: {hit.entity}")
    ```

**Example with `expr` and specific `output_fields`**

You can also filter search results using an expression and retrieve only specific fields.

**带有 `expr` 和特定 `output_fields` 的示例**

您还可以使用表达式过滤搜索结果，并仅检索特定字段。

```python
search_params = {
    "metric_type": "IP",  # Inner Product
    "params": {"nprobe": 10},
}

results = collection.search(
    data=[[0.1, 0.2, 0.3]],
    anns_field="vector",
    param=search_params,
    limit=5,
    expr="genre == 'Sci-Fi'",  # Filter by genre
    output_fields=["title", "year"]  # Retrieve only title and year
)

for hit in results[0]:
    print(f"Hit ID: {hit.id}, Score: {hit.score}, Title: {hit.entity.get('title')}, Year: {hit.entity.get('year')}")
```

```python
search_params = {
    "metric_type": "IP",  # 内积
    "params": {"nprobe": 10},
}

results = collection.search(
    data=[[0.1, 0.2, 0.3]],
    anns_field="vector",
    param=search_params,
    limit=5,
    expr="genre == 'Sci-Fi'",  # 按类型过滤
    output_fields=["title", "year"]  # 仅检索标题和年份
)

for hit in results[0]:
    print(f"命中 ID: {hit.id}, 评分: {hit.score}, 标题: {hit.entity.get('title')}, 年份: {hit.entity.get('year')}")
```

**Explanation of parameters**

*   `data`: A list of search vectors.
*   `anns_field`: The name of the vector field to search against.
*   `param`: A dictionary containing search parameters, such as `metric_type` and `params`.
    *   `metric_type`: The distance metric used for similarity calculation (e.g., "L2", "IP", "COSINE").
    *   `params`: Index-specific parameters. For IVF indexes, `nprobe` is a common parameter that controls the trade-off between search accuracy and speed.
*   `limit`: The maximum number of results to return for each search vector.
*   `expr`: An optional expression to filter the search results based on scalar field values.
*   `output_fields`: A list of fields to return along with the search results. If set to `["*"]`, all fields are returned.

**参数说明**

*   `data`: 搜索向量的列表。
*   `anns_field`: 要搜索的向量字段的名称。
*   `param`: 包含搜索参数的字典，例如 `metric_type` 和 `params`。
    *   `metric_type`: 用于相似度计算的距离度量（例如 "L2"、"IP"、"COSINE"）。
    *   `params`: 特定于索引的参数。对于 IVF 索引，`nprobe` 是一个常用参数，用于控制搜索精度和速度之间的权衡。
*   `limit`: 每个搜索向量要返回的最大结果数。
*   `expr`: 一个可选的表达式，用于根据标量字段值过滤搜索结果。
*   `output_fields`: 要随搜索结果一起返回的字段列表。如果设置为 `["*"]`，则返回所有字段。

**Note:**

The scores returned by Milvus are dependent on the `metric_type` used. For example:

*   **L2 (Euclidean distance):** Lower scores indicate higher similarity.
*   **IP (Inner Product):** Higher scores indicate higher similarity.
*   **COSINE:** Scores range from -1 to 1, where 1 indicates maximum similarity.

**注意：**

Milvus 返回的评分取决于所使用的 `metric_type`。例如：

*   **L2（欧氏距离）：** 分数越低表示相似度越高。
*   **IP（内积）：** 分数越高表示相似度越高。
*   **COSINE：** 分数范围从 -1 到 1，其中 1 表示最大相似度。

返回的距离分数在 0-1 之间。0 表示不相似，1 表示最相似。

In [ ]:
docs = db.similarity_search_with_score(query)

In [4]:
print(docs[0])

(Document(page_content='And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.', metadata={'source': '../../how_to/state_of_the_union.txt'}), 0.561813814013747)


## 恢复之前创建的表和添加的数据

AwaDB 会自动持久化已添加的文档数据。

如果您可以恢复之前创建和添加的表，您可以这样做：

In [ ]:
import awadb

awadb_client = awadb.Client()
ret = awadb_client.Load("langchain_awadb")
if ret:
    print("awadb load table success")
else:
    print("awadb load table failed")